# How the District buys software

Everything in the briefing comes from cells in this notebook. Each section states where
the number comes from and what it does not cover.

Source: the DC Contracts and Procurement Transparency Portal, <https://contracts.ocp.dc.gov>.
The portal publishes contracts, purchase orders, payments and solicitations for agencies
that buy through the Office of Contracting and Procurement. Independent agencies are not
in it. Neither is anything bought below the posting threshold.

To reproduce: `uv run python -c "from dcprocure.pull import pull_all; pull_all(all_classes=True)"`,
then `uv run python analysis/classify_contracts.py` and `analysis/classify_delivery.py`.
The pull writes a manifest recording the date, because the portal is live and the counts
move between pulls. A resumed pull can leave the manifest describing an earlier, smaller
snapshot; `analysis/rebuild_manifest.py` rebuilds it from the files on disk.


In [1]:
import sys, json; sys.path.insert(0, '../src')
import pandas as pd
from dcprocure import parse
pd.set_option('display.width', 200)

contracts = parse.load_contracts()
orders    = parse.load_purchase_orders()
payments  = parse.load_payments()
labels    = pd.read_csv('../data/contract_classifications.csv')
delivery  = pd.read_csv('../data/delivery_structure.csv')

manifest = json.load(open('../data/raw/manifest.json'))
print('pulled on', manifest['pulled_on'])
print(f'{len(contracts):,} contract option periods, {contracts.contract_number.nunique():,} distinct contracts')
print(f'{len(orders):,} purchase orders, {len(payments):,} payments')


pulled on 2026-08-24
7,113 contract option periods, 3,983 distinct contracts
114,255 purchase orders, 866,823 payments


## A contract row is an option period, not a contract

This trips up every analysis of this data. A five-year contract appears once per option
period, each row carrying that period's own amount. Summing the rows gives a lifetime
ceiling. It is not annual spending and it is not money that changed hands.


In [2]:
per_contract = contracts.groupby('contract_number').size()
print(f'{len(contracts):,} rows describe {contracts.contract_number.nunique():,} contracts')
print(f'most option periods on one contract: {per_contract.max()}')
contracts[contracts.contract_number == per_contract.idxmax()][
    ['contract_number','option_period','amount','start_date','end_date']
].sort_values('start_date')


7,113 rows describe 3,983 contracts
most option periods on one contract: 6


,contract_number,option_period,amount,start_date,end_date
4154,CW88488,Base Period,956401.0,2021-01-01,2021-12-31
4136,CW88488,Option 1,985093.0,2022-01-01,2022-12-31
4121,CW88488,Option 2,1014646.0,2023-01-01,2023-12-31
4107,CW88488,Option 3,1045085.0,2024-01-01,2024-12-31
4095,CW88488,Option 4,1076438.0,2025-01-01,2025-12-31
4076,CW88488,Option 5,923943.0,2026-01-01,2026-10-31


## What the District is buying

NIGP commodity codes cannot answer this. They are self-assigned and inconsistent: class 918
("consulting") holds Medicaid managed care contracts worth billions, class 920 ("data
processing, computer, programming and software services") holds a plumbing contractor, and
the District's largest IT staffing vehicles sit in 961/962, "miscellaneous services, not
otherwise classified".

So each contract was classified from its title, vendor and awarding agency by an LLM with a
constrained schema. Prompt and schema live in `classifier_config.yaml`. Accuracy against 20
hand-labelled contracts in `tests/gold_labels.csv` is 20/20; run `tests/eval_labels.py` to
check. The labelled cases were chosen because commodity codes get them wrong in both
directions.


In [3]:
tech = labels[labels.is_technology == True]
table = labels.groupby('work_type').agg(contracts=('contract_number','size'), ceiling=('ceiling','sum'))
table['ceiling_M'] = (table.ceiling/1e6).round(0)
print(f'{len(tech):,} technology contracts, ${tech.ceiling.sum()/1e6:,.0f}M in ceilings')
table[['contracts','ceiling_M']].sort_values('ceiling_M', ascending=False)


1,372 technology contracts, $4,641M in ceilings


,contracts,ceiling_M
work_type,,
not_technology,2611,57426.0
ordering_vehicle,328,1840.0
staff_augmentation,211,1203.0
hardware_or_license_resale,262,502.0
it_consulting_advisory,165,332.0
cots_implementation,125,291.0
software_om_support,126,227.0
telecom_or_network,95,208.0
custom_software_build,60,38.0


Custom software development is the smallest category. Staff augmentation carries thirty
times its ceilings, and ordering vehicles — schedule contracts that authorise future task
orders and buy nothing themselves — carry nearly fifty times.


In [4]:
for wt in ['ordering_vehicle','staff_augmentation','custom_software_build']:
    rows = labels[labels.work_type == wt]
    print(f'{wt:<24} {len(rows):>4} contracts   ${rows.ceiling.sum()/1e6:>7,.0f}M')


ordering_vehicle          328 contracts   $  1,840M
staff_augmentation        211 contracts   $  1,203M
custom_software_build      60 contracts   $     38M


## Ceilings are not spending

Contract amounts are authorisations. Comparing them against recorded payments shows how
far apart those are, and how differently contract types behave.

Read the absolute percentages as floors. Payments join to contracts by contract number, and
a task order issued under an ordering vehicle is paid under the task order's own number, so
it never joins back to the parent. Only 44% of technology contracts have any linked payment.


In [5]:
paid = payments.groupby('contract_number')['amount'].sum()
t = tech.copy()
t['paid'] = t.contract_number.map(paid).fillna(0.0)
t['contract_type'] = t.contract_number.map(contracts.groupby('contract_number').contract_type.first())
d = t.groupby(t.contract_type.fillna('unknown')).agg(
        n=('contract_number','size'), ceiling=('ceiling','sum'), paid=('paid','sum'))
d['drawdown_pct'] = (100*d.paid/d.ceiling).round(1)
d['ceiling_M'] = (d.ceiling/1e6).round(0); d['paid_M'] = (d.paid/1e6).round(0)
print(f'technology contracts with any linked payment: {(t.paid>0).sum()} of {len(t)} ({100*(t.paid>0).mean():.0f}%)')
d[['n','ceiling_M','paid_M','drawdown_pct']].sort_values('ceiling_M', ascending=False)


technology contracts with any linked payment: 603 of 1372 (44%)


,n,ceiling_M,paid_M,drawdown_pct
contract_type,,,,
Indefinite Delivery Indefinite Quantity,480,2968.0,112.0,3.8
Firm Fixed Price,706,1165.0,518.0,44.5
Labor Hour,125,436.0,173.0,39.7
Requirements,45,58.0,33.0,57.2
Time And Materials,4,7.0,1.0,16.5
unknown,8,5.0,4.0,83.0
Fixed Price with Economic Price Adjustment,2,1.0,0.0,13.5
Letter Contract,1,1.0,0.0,29.5
Fixed Price Incentive,1,0.0,0.0,110.8


## Two defects in the portal

Both produce wrong answers silently, and both were found by testing the API against itself.

The fiscal-year filter on contract search does nothing. Send any year and the server returns
the entire contract universe, identical for every year, while reporting that it applied no
filters at all. Anyone charting year-over-year change from the portal's own filter is
charting the same number repeatedly. The client in `dcprocure/client.py` compares the
filters the server echoes against what was sent and raises rather than return a wider slice
than asked for.


In [6]:
from dcprocure.client import DCProcurementClient, Filter, PortalError
with DCProcurementClient() as client:
    try:
        client.search('contracts', [Filter('FiscalYear','2025')])
        print('no guard fired')
    except PortalError as exc:
        print(exc)


contracts search silently ignored filter value(s) ['2025']; the server applied only []. Results would cover a wider slice than requested.


The second defect is worse because it is invisible. `paymentAmount` is not the amount of
that payment: it is the total of the whole check or ACH run the voucher was paid in,
repeated on every row belonging to that run. Summing it inflates the total more than
threefold, and the error concentrates on vendors paid in large batched runs.

The clearest case is CW95481, an OCTO IT staffing vehicle with a $120M ceiling.


In [7]:
star = payments[payments.contract_number == 'CW95481']
print(f'{len(star):,} payment rows on a contract with a $120M ceiling')
print(f'summing voucherAmount (correct): ${star.amount.sum()/1e6:,.0f}M')
print(f'summing paymentAmount (wrong):   ${star.check_total.sum()/1e9:,.1f}B')
print()
print(f'across the whole table: ${payments.amount.sum()/1e9:.1f}B correct vs ${payments.check_total.sum()/1e9:.1f}B wrong')
print(f'payments carrying no contract number at all: {payments.contract_number.isna().sum():,}',
      f'(${payments[payments.contract_number.isna()].amount.sum()/1e9:.1f}B)')


11,121 payment rows on a contract with a $120M ceiling
summing voucherAmount (correct): $132M
summing paymentAmount (wrong):   $17.3B

across the whole table: $33.3B correct vs $113.7B wrong


payments carrying no contract number at all: 478,716 ($17.8B)


## Ten vendors take half the traceable technology payments

Of the technology payments that can be tied to a contract between FY2021 and FY2025,
half went to ten vendors and 17% to one. The basis matters: this is measured on the
44% of technology contracts whose payments carry a matching contract number. The
unlinked majority, including everything bought through the DC Supply Schedule, could
be more or less concentrated. The top vendor does not depend on the join: the
purchase-order table, a separate path into the data, also puts Computer Aid first.

Vendor names are normalised first, because the portal records CDW Government three
ways. The merge rules and every pair they combine are in `vendor_aliases.yaml`.


In [8]:
window = payments[payments.fiscal_year.between(2021, 2025)
                  & payments.contract_number.isin(set(tech.contract_number))].copy()
window['vendor_key'] = window.vendor.map(parse.normalize_vendor)
total = window.amount.sum()
ranked = window.groupby('vendor_key').amount.sum().sort_values(ascending=False)
print(f"${total/1e6:,.0f}M traceable technology payments, {ranked.size} vendors after normalising")
for n in (1, 5, 10):
    print(f"top {n:>2} vendors: {100*ranked.head(n).sum()/total:.1f}%")
(ranked.head(10)/1e6).round(1).to_frame('paid_M')


$579M traceable technology payments, 295 vendors after normalising
top  1 vendors: 17.4%
top  5 vendors: 42.3%
top 10 vendors: 52.5%


,paid_M
vendor_key,
COMPUTER AID,100.6
CDW GOVERNMENT,55.1
KPMG,37.1
CARAHSOFT TECHNOLOGY,33.9
ACCENTURE,17.8
GEOGRAPHIC SOLUTIONS,14.8
INTERNATIONAL BUSINESS MACHINE,12.0
RIDGE IT CYBER,11.4
QUICKBASE,10.8


## Where the largest IT vehicle sits

Computer Aid, Inc. is the District's biggest IT services vendor by dollars. Its contracts
are coded NIGP 962, "miscellaneous services, not otherwise classified", and include one
titled "Information Technology Staff Augmentation Services". The purchase orders written
against those contracts are coded 918, "consulting". Neither is a technology code, so an
analysis filtered to IT commodity classes misses the vendor entirely.

The shape of the purchase orders is the point: about 1,200 a year at a median near $60,000,
spread across 33 agencies, eight of which take 85% of the dollars. That is one
person-period per order.


In [9]:
recent = orders[orders.fiscal_year.between(2022, 2026)].copy()
recent['vendor_key'] = recent.vendor.str.upper().str.strip()
cai = recent[recent.vendor_key == 'COMPUTER AID  INC']
print(f'FY2022-FY2026: {len(cai):,} purchase orders, ${cai.amount.sum()/1e6:,.0f}M, median ${cai.amount.median():,.0f}')
print()
comparison = recent[recent.vendor_key.isin(['COMPUTER AID  INC','FAST ENTERPRISES LLC','CDW GOVERNMENT LLC'])]
comparison.groupby('vendor_key')['amount'].agg(orders='size', total='sum', median='median').assign(
    total_M=lambda x: (x.total/1e6).round(1)).drop(columns='total')


FY2022-FY2026: 5,787 purchase orders, $465M, median $59,944



,orders,median,total_M
vendor_key,,,
CDW GOVERNMENT LLC,374,25020.855,89.0
COMPUTER AID INC,5787,59944.000,465.4
FAST ENTERPRISES LLC,11,3513794.860,51.1


## Published hourly rates

42 of the 359 technology contracts with a readable award document publish an hourly rate
card in their price schedule. `analysis/extract_rates.py` pulls them out; every rate is
re-found verbatim in the award text before it is kept. Matching a price to its job title
is the fragile part — some PDFs scramble their tables into separate column runs — so each
row also records whether the title-to-price pairing could be verified (deterministic
layout reconstruction for CW91684, text proximity elsewhere). Pairings that could not be
verified are published as price ranges only.


In [10]:
rates = pd.read_csv('../data/labor_rates.csv')
ok = rates[rates.pairing.isin(['deterministic', 'proximate'])]
print(f"{rates.contract_number.nunique()} contracts publish {len(rates):,} rates; "
      f"{len(ok):,} have a verified title-to-price pairing")
print(f"min ${rates.hourly_rate.min():.2f}   median ${rates.hourly_rate.median():.2f}   "
      f"max ${rates.hourly_rate.max():.2f}")
dev = ok[ok.labor_category.str.contains('develop|programmer|software', case=False)]
dev = dev.drop_duplicates(subset=['contract_number', 'labor_category', 'level', 'hourly_rate'])
dev.groupby(['vendor', 'labor_category']).hourly_rate.agg(['min', 'max']).round(2)


42 contracts publish 2,146 rates; 1,304 have a verified title-to-price pairing
min $24.01   median $150.00   max $1015.00


min     max
vendor                                        labor_category                                                    
AVID SYSTEMS, LLC                             System Architect (Software/Data/Database) I         140.00  140.00
CENTER FOR EDUCATION AND EMPLO                Software Engineer                                   109.51  109.51
                                              Software Engineer (subcontractor)                    95.23   95.23
COMPUTER AID, INC.                            Geographic Information System (GIS) Analyst/Pro...   33.42   87.24
                                              Mobile App Developer                                 70.76   94.89
                                              Web Developer                                        55.98   77.00
District Consultant Limited Liability Company Category Developer                                  144.75  144.75
EASTBANC TECHNOLOGIES                         Database Developer                                  180.32  220.39
                                              Developer II                                        184.83  194.18
                                              Developer II (Open Market)                          199.03  204.01
Karder Corporation                            Database Developer                                  157.50  180.11
                                              Developer                                           182.33  189.12
LIMBIC SYSTEMS INC                            Software Engineer                                   135.00  135.00

## Agile language

Counting keywords does not work here, and getting this wrong produced a finding that later
had to be withdrawn. A first pass counted "scrum", "sprint" and "backlog" anywhere in the
contract text and found what looked like a solid association with waterfall structure.
Hand-reading the flagged contracts found that five of thirteen were false
positives: three matched "Experience in Agile methodology such as Scrum, Kanban, or Scaled
Agile Framework" inside labor-category staff qualifications, one matched a "backlog of cases
pending adjudication" at an administrative hearings office, and one matched "Support One
Sprint" as a priced line item. Removing them left nothing. A separate bug made it worse:
matching `\bSAFe\b` case-insensitively also matches the ordinary word "safe".

The rewritten pass reads each mention in context and asks what the words are doing there.
Accuracy against 20 hand-read contracts in `tests/delivery_gold_labels.csv` is 20/20 on
structure and 5/5 on rejecting the false-positive traps; it identifies 4 of 6 genuine
commitments, so it undercounts. The two commitments it misses are corrected by hand in
`delivery_overrides.yaml`, with the reasons recorded there. Run `tests/eval_delivery.py`
to check.


In [11]:
mentions = delivery[delivery.agile_mention_kind != 'no_agile_language']
print(f'{len(delivery)} contracts with readable award text; {len(mentions)} mention agile at all')
counts = delivery.agile_mention_kind.value_counts()
pd.DataFrame({'contracts': counts, 'pct_of_all': (100*counts/len(delivery)).round(1)})


288 contracts with readable award text; 38 mention agile at all


,contracts,pct_of_all
agile_mention_kind,,
no_agile_language,250,86.8
personnel_qualification,14,4.9
delivery_commitment,11,3.8
unrelated_usage,10,3.5
aspirational_only,2,0.7
pricing_unit,1,0.3


Most agile language in DC contracts is not a commitment to work that way. More of it
appears in staff qualification requirements than in delivery commitments.


In [12]:
commits = delivery[delivery.agile_mention_kind == 'delivery_commitment']
print(f'{len(commits)} of {len(mentions)} mentions are actual delivery commitments',
      f'({100*len(commits)/len(mentions):.0f}%)')
print()
print(pd.crosstab(delivery.delivery_structure,
                  delivery.agile_mention_kind == 'delivery_commitment',
                  margins=True).to_string())


11 of 38 mentions are actual delivery commitments (29%)



agile_mention_kind             False  True  All
delivery_structure                             
goods_or_licenses                 51     0   51
incremental_capacity              36     2   38
milestone_payments_only            4     0    4
recurring_operations              76     0   76
sequential_gated_deliverables     73     9   82
structure_not_stated              37     0   37
All                              277    11  288


Of the eleven contracts that do commit to agile delivery, nine are structurally sequential
and gated: fixed scope, deliverables accepted at stages, payment following acceptance. Two
of the eleven are hand-read corrections of classifier misses, recorded in
`delivery_overrides.yaml`.

Eleven contracts is a small number. The quotes below carry more weight than the count.


In [13]:
gated = (commits.delivery_structure == 'sequential_gated_deliverables').sum()
rest = delivery[delivery.agile_mention_kind != 'delivery_commitment']
rest_gated = (rest.delivery_structure == 'sequential_gated_deliverables').sum()
print(f'commit to agile and structurally gated: {gated}/{len(commits)} ({100*gated/len(commits):.0f}%)')
print(f'everything else:                        {rest_gated}/{len(rest)} ({100*rest_gated/len(rest):.0f}%)')
print()
for r in commits.sort_values('ceiling', ascending=False).itertuples():
    print(f'${r.ceiling/1e6:6.2f}M  {r.delivery_structure}')
    print(f'         {str(r.title)[:64]}')
    print(f'         {str(r.agile_quote)[:150]}')
    print()


commit to agile and structurally gated: 9/11 (82%)
everything else:                        73/277 (26%)

$  7.92M  sequential_gated_deliverables
         ROFTOP - Clinical Case Management System
         using a hybrid waterfall and Agile methodology. The Contractor shall demonstrate how it would use this methodology by submitting its proposed process,

$  2.35M  incremental_capacity
         IT Infrastructure Support and Management Consulting Services
         C.5.6.3 The Contractor shall manage all agile processes using tools available to DBH stakeholders. At a minimum, the Contractor shall: a) Maintain ITE

$  2.13M  sequential_gated_deliverables
         Upgrading and Maintenance Offender Management System
         "During the UAT phase, the ViaPath engineering team will operate within ViaPath’s existing Scrum Agile framework, using two-week sprints to iterativel

$  2.08M  sequential_gated_deliverables
         Unemployment Insurance Claimant Portal
         "The Contractor shall 

## Half the technology ceiling authority sits on one schedule

An ordering vehicle authorises future task orders without buying anything itself.
Nearly all of the District's are one programme: the DC Supply Schedule, awarded in
near-identical form to a pool of pre-qualified vendors. The dollars below are ceilings
summed across that pool, not money that moves, and most of the authority will never
collectively be spent.

The portal has no record type for task orders placed against these vehicles. Some
schedule buying surfaces later as purchase orders or further contracts, but nothing
links those back to their vehicle, so the step where the actual buying decision
happens cannot be followed in this data.


In [14]:
dcss = tech[tech.title.str.contains('DCSS', case=False, na=False)]
vehicles = tech[tech.work_type == 'ordering_vehicle']
print(f'technology contracts naming DCSS: {len(dcss)}, ${dcss.ceiling.sum()/1e6:,.0f}M in ceilings')
print(f'that is {100*dcss.ceiling.sum()/tech.ceiling.sum():.0f}% of all technology ceilings')
print(f'ordering vehicles: {len(vehicles)}, of which DCSS: '
      f"{vehicles.title.str.contains('DCSS', case=False, na=False).sum()}")


technology contracts naming DCSS: 397, $2,410M in ceilings
that is 52% of all technology ceilings
ordering vehicles: 328, of which DCSS: 305


## Named systems

273 of 1,372 technology contracts name a specific system in the title, so every count
here is a floor. The names show what the money maintains. A handful of large systems absorb repeated contracts over years;
no single contract builds any of them.


In [15]:
named = tech[tech.system_name.notna() & (tech.system_name.astype(str).str.strip() != '')]
named = named[named.system_name.astype(str).str.lower() != 'nan']
print(f'{len(named)} of {len(tech)} technology contracts name a system')
by_system = named.groupby(named.system_name.astype(str).str.upper().str.strip()).agg(
    contracts=('contract_number','size'), ceiling=('ceiling','sum'), agency=('agency','first'))
by_system['ceiling_M'] = (by_system.ceiling/1e6).round(1)
by_system[['contracts','ceiling_M','agency']].sort_values('ceiling_M', ascending=False).head(10)


273 of 1372 technology contracts name a system


,contracts,ceiling_M,agency
system_name,,,
DCAS,28,198.3,Health Care Finance (DHCF)
MEDICAID MANAGEMENT INFORMATION SYSTEM (MMIS),2,174.3,Health Care Finance (DHCF)
CAPITAL BIKESHARE,1,50.0,Transportation (DDOT)
MOBIS,4,32.0,Contracting and Procurement (OCP)
UIBS,2,14.8,Employment Services (DOES)
SAP CLOUD,1,13.0,Contracting and Procurement (OCP)
DOCS,1,10.0,Employment Services (DOES)
NEXT GENERATION 9-1-1 (NG9-1-1),1,9.0,Unified Communications (OUC)
ROFTOP - CLINICAL CASE MANAGEMENT SYSTEM,1,7.9,Health Care Finance (DHCF)


DCAS is the clearest example: at least 28 separate contracts for operations and maintenance,
application development teams, licences, hosting, printing and mailing.


In [16]:
dcas = named[named.system_name.astype(str).str.upper().str.strip() == 'DCAS']
dcas[['contract_number','title','vendor','ceiling','work_type']].sort_values(
    'ceiling', ascending=False).head(12)


,contract_number,title,vendor,ceiling,work_type
1396,CW121488,FY25 - DCAS - Operations Maintenance Services,D2Sol Inc,83697969.00,software_om_support
3154,CW87331,DCAS Application Development Team,CDE Systems,31649600.00,staff_augmentation
2615,CW67170,DCAS R3 IV&V,"Accenture, LLP",23271000.00,it_consulting_advisory
3104,CW85955,DCAS IBM Curam Software and Support,International Business Machine,11662800.00,software_om_support
3417,CW91846,DCAS Temp Resource - R3 Deployment - Incumbent...,"Enlightened, Inc.",8036707.72,staff_augmentation
2516,CW62227,DCAS R3,"KPMG, LLP",7281205.00,it_consulting_advisory
2560,CW64904,DCAS - Temporary Personnel - Avid Systems,"AVID SYSTEMS, LLC",7270220.00,staff_augmentation
63,CW100807,DCAS - Mobile Application Support,"Diona US, Inc.",6618400.00,software_om_support
1001,CW114997,DCAS Amazon Web Services (Carahsoft),Carahsoft Technology Corporation,4072457.96,hardware_or_license_resale
2830,CW77036,DCAS - Temporary Personnel - Cradle Systems,Cradle Systems,2609473.48,staff_augmentation


## What this does not cover

Independent agencies procure separately and are not in the portal.

Task orders issued under ordering vehicles are not published, so the work actually bought
through the DC Supply Schedule is invisible here. That is a large hole: schedule vehicles
carry the biggest ceilings in the technology set.

Delivery structure could only be read for 288 of 476 candidate contracts. The rest had no
award document attached, or a scanned one with no text layer. Scanned awards would need OCR.

Payments join to contracts for 44% of technology contracts. The remaining spending is real
but cannot be attributed to a contract by number.

Contract classification is done by a language model reading titles, vendors and agencies,
not contract text. It is tested against 20 hand-read labels chosen to be hard, and the
test suite is in the repository, but twenty cases bound the error loosely. Treat category
totals as approximate; the ordering between categories is what survives.
